In [4]:
pip install faiss-cpu streamlit google-generativeai

INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of proto-plus to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 95.1 MB/

In [26]:
from google.colab import userdata
userdata.get('GEMINI_API_KEY')

'AQ.Ab8RN6K9QEzD8nnzR1RHxq5x3_Yh-vMcCBa9PLKCfG6eefE33Q'

In [47]:
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

'AQ.Ab8RN6KqHIru88DwVNaoZ3sbGwcJ1h9jLks1Pb5kkXnjehRwCw'

In [18]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone

import faiss
import numpy as np
import pandas as pd
import streamlit as st
import yfinance as yf
from google import genai
from dotenv import load_dotenv

In [19]:
load_dotenv()

DATA_DIR = Path("financial_kb")
DATA_DIR.mkdir(exist_ok=True)

EMBED_MODEL = "gemini-embedding-001"
GEN_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

CHUNK_SIZE = 1200
OVERLAP = 150

st.set_page_config(
    page_title="AI Financial Analyst",
    page_icon="📈",
    layout="wide"
)

st.title("AI Financial Analyst")
st.caption(
    "Yahoo Finance + FAISS + Google Gemini RAG"
)


2026-10-12 19:45:26.813 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:45:26.816 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:45:26.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:45:26.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:45:26.821 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:45:26.823 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:45:26.824 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [20]:
api_key = os.getenv("GEMINI_API_KEY")

In [29]:
def collect_financial_data(symbol):
    stock = yf.Ticker(symbol)

    try:
        info = stock.get_info()
    except Exception:
        info = {}

    documents = []

    # Company information and ratios
    profile = [
        f"Company ticker: {symbol}",
        f"Collection date: {datetime.now(timezone.utc).isoformat()}"
    ]

    fields = [
        "longName", "sector", "industry", "currency",
        "marketCap", "totalRevenue", "revenueGrowth",
        "netIncomeToCommon", "trailingEps", "forwardEps",
        "trailingPE", "forwardPE", "priceToBook",
        "profitMargins", "operatingMargins", "returnOnEquity",
        "returnOnAssets", "debtToEquity", "currentRatio",
        "quickRatio", "totalCash", "totalDebt",
        "operatingCashflow", "freeCashflow"
    ]

    for field in fields:
        value = info.get(field)
        if value is not None:
            profile.append(f"{field}: {value}")

    documents.append({
        "source": "Company Profile and Key Metrics",
        "text": "\n".join(profile)
    })

    # Financial statements
    statements = [
        ("Income Statement", stock.income_stmt),
        ("Balance Sheet", stock.balance_sheet),
        ("Cash Flow Statement", stock.cashflow)
    ]

    for name, frame in statements:
        if frame is None or frame.empty:
            text = f"{name}: Data unavailable."
        else:
            text = (
                f"{name} for {symbol}\n"
                + frame.to_string()
            )

        documents.append({
            "source": name,
            "text": text
        })

    # Historical prices
    history = stock.history(
        period="5y",
        auto_adjust=False
    )

    if not history.empty:
        monthly = history.resample("ME").agg({
            "Open": "first",
            "High": "max",
            "Low": "min",
            "Close": "last",
            "Volume": "sum"
        })

        documents.append({
            "source": "Historical Stock Prices",
            "text": (
                f"Historical prices for {symbol}\n"
                + monthly.to_string()
            )
        })

    return documents

In [30]:

def chunk_text(text):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))

        # Prefer a line boundary when possible
        if end < len(text):
            boundary = text.rfind("\n", start, end)
            if boundary > start + CHUNK_SIZE // 2:
                end = boundary

        piece = text[start:end].strip()

        if piece:
            chunks.append(piece)

        if end >= len(text):
            break

        start = max(end - OVERLAP, start + 1)

    return chunks

In [31]:
def generate_embeddings(texts, task_type):
    vectors = []

    for start in range(0, len(texts), 16):
        batch = texts[start:start + 16]

        result = client.models.embed_content(
            model=EMBED_MODEL,
            contents=batch,
            config={
                "task_type": task_type,
                "output_dimensionality": 768
            }
        )

        vectors.extend([
            item.values for item in result.embeddings
        ])

    vectors = np.asarray(vectors, dtype="float32")
    faiss.normalize_L2(vectors)

    return vectors

In [32]:
def build_knowledge_base(symbol):
    documents = collect_financial_data(symbol)

    chunks = []

    for document in documents:
        for number, part in enumerate(
            chunk_text(document["text"]), start=1
        ):
            chunks.append({
                "source": document["source"],
                "chunk_number": number,
                "text": part
            })

    if not chunks:
        raise ValueError("No financial information was collected.")

    texts = [
        f"Source: {item['source']}\n{item['text']}"
        for item in chunks
    ]

    embeddings = generate_embeddings(
        texts,
        "RETRIEVAL_DOCUMENT"
    )

    # Inner product on normalized vectors = cosine similarity
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    folder = DATA_DIR / symbol
    folder.mkdir(parents=True, exist_ok=True)

    faiss.write_index(
        index,
        str(folder / "index.faiss")
    )

    with open(folder / "chunks.json", "w", encoding="utf-8") as file:
        json.dump(chunks, file, indent=2)

    with open(folder / "metadata.json", "w", encoding="utf-8") as file:
        json.dump({
            "ticker": symbol,
            "chunk_count": len(chunks),
            "created_at": datetime.now(timezone.utc).isoformat(),
            "embedding_model": EMBED_MODEL
        }, file, indent=2)

    return folder, len(chunks)

In [33]:
def retrieve_documents(folder, question, top_k=5):
    index = faiss.read_index(
        str(folder / "index.faiss")
    )

    with open(folder / "chunks.json", encoding="utf-8") as file:
        chunks = json.load(file)

    query_vector = generate_embeddings(
        [f"task: question answering | query: {question}"],
        "RETRIEVAL_QUERY"
    )

    scores, indices = index.search(
        query_vector,
        min(top_k, len(chunks))
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            result = chunks[int(idx)].copy()
            result["similarity"] = float(score)
            results.append(result)

    return results

In [34]:
def answer_question(folder, symbol, question):
    retrieved = retrieve_documents(folder, question)

    context = "\n\n".join(
        f"[Source {i + 1}: {item['source']}]\n"
        f"{item['text']}"
        for i, item in enumerate(retrieved)
    )

    prompt = f"""
You are an educational financial analyst for {symbol}.

Answer the question using ONLY the retrieved financial context.

Rules:
1. Never invent financial figures or dates.
2. If evidence is missing, explicitly say so.
3. Cite factual claims using [Source 1], [Source 2], etc.
4. State reporting periods and units when available.
5. Explain calculations and show formulas.
6. Separate reported facts from derived calculations.
7. Do not provide personalized investment advice.

Question:
{question}

Retrieved Financial Context:
{context}
"""

    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt
    )

    return response.text, retrieved

In [42]:
ticker = st.sidebar.text_input("Stock Ticker", value="AAPL").strip().upper()

if not ticker:
    st.warning("Please enter a valid stock ticker in the sidebar.")
    st.stop()

folder = DATA_DIR / ticker

if st.sidebar.button("Collect Data and Build Index"):
    try:
        with st.spinner("Collecting data and building FAISS index..."):
            folder, count = build_knowledge_base(ticker)

        st.success(
            f"Knowledge base created: {count} chunks indexed."
        )

    except Exception as error:
        st.error(f"Could not build index: {error}")

if not (folder / "index.faiss").exists():
    st.warning(
        f"No knowledge base exists for {ticker}. "
        "Click 'Collect Data and Build Index' first."
    )
    st.stop()

question = st.text_input(
    "Ask a question about the company",
    placeholder="What are the company's revenue, debt, and cash flow trends?"
)

if st.button("Generate Financial Analysis") and question.strip():
    try:
        with st.spinner("Searching financial data and generating response..."):
            answer, evidence = answer_question(
                folder, ticker, question
            )

        st.subheader("Financial Analysis")
        st.markdown(answer)

        st.subheader("Retrieved Evidence")

        for i, item in enumerate(evidence, start=1):
            with st.expander(
                f"Source {i}: {item['source']} "
                f"(similarity: {item['similarity']:.3f})"
            ):
                st.write(item["text"])

    except Exception as error:
        st.error(f"Error: {error}")

st.caption(
    "Data from Yahoo Finance. Verify figures against official filings. "
    "This application is for educational use only."
)

2026-10-12 19:55:16.362 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:55:16.365 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:55:16.366 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:55:16.368 Session state does not function when running a script without `streamlit run`
2026-10-12 19:55:16.370 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:55:16.371 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:55:16.372 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:55:16.374 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 19:55

DeltaGenerator()

In [50]:
from google.colab import userdata
try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.getenv("GEMINI_API_KEY")

clt = genai.Client(api_key=api_key)
response = clt.models.generate_content(
    model="gemini-3.8-flash",
    contents="What is Apple's latest available annual revenue?"
)
print(response.text)

For its most recent full fiscal year (Fiscal Year 2024, which ended on September 28, 2024), Apple reported a total annual revenue of **$391.04 billion** ($391,035 million). 

This represents an increase of approximately 2% compared to the **$383.29 billion** reported in Fiscal Year 2023.


In [51]:
clt = genai.Client(api_key=api_key)
response2 = clt.models.generate_content(
    model="gemini-3.8-flash",
    contents="How has Apple's revenue changed over the available years?"
)
print(response2.text)

Apple’s revenue trajectory is one of the most remarkable corporate growth stories in modern business history. Over its nearly 50-year history, Apple transformed from a niche computer maker near bankruptcy in the late 1990s into a global technology powerhouse generating nearly **$400 billion** in annual revenue.

Here is a breakdown of how Apple’s revenue has changed over the key eras of its history, followed by major milestones and core revenue drivers.

---

### 1. Chronological Evolution

#### Phase 1: Early Growth and Near-Collapse (1977–1997)
* **1977–1984:** Driven by the Apple II, revenue grew rapidly from **$773,000 in 1977** to over **$1.5 billion by 1984**, the year the original Macintosh was introduced.
* **Late 1980s to Mid-1990s:** Revenue peaked around **$11 billion in 1995**, but intense competition from Microsoft/Windows and poor internal execution led to product failures and market share loss.
* **1996–1997 (Near Bankruptcy):** By 1997, revenue had plunged to **$7.1 bil

In [54]:

import json
from datetime import datetime

# Enter your conversation messages here
chat_history = [
    {
        "role": "user",
        "message": "Build a RAG-based AI Financial Analyst project."
    },
    {
        "role": "assistant",
        "message": "The project uses Python, Streamlit, Gemini API, FAISS, and Yahoo Finance."
    }
]

# Create a filename with the current date and time
Chatbot_AI = datetime.now().strftime("chat_history_%Y%m%d_%H%M%S.txt")

# Save the conversation
with open(Chatbot_AI, "w", encoding="utf-8") as file:
    for chat in chat_history:
        file.write(f'{chat["role"].upper()}:\n')
        file.write(chat["message"] + "\n\n")

print(f"Chat history saved successfully to {Chatbot_AI}")

Chat history saved successfully to chat_history_20261012_200306.txt


In [55]:
print(chat_history)

[{'role': 'user', 'message': 'Build a RAG-based AI Financial Analyst project.'}, {'role': 'assistant', 'message': 'The project uses Python, Streamlit, Gemini API, FAISS, and Yahoo Finance.'}]


In [60]:
# 1. Initialize the global client used inside generate_embeddings()
from google.colab import userdata
try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

# 2. Set the global model config used by answer_question()
GEN_MODEL = "gemini-3.8-flash"

def ask_question(question, top_k=5):
    ticker_symbol = "AAPL"
    folder_path = DATA_DIR / ticker_symbol

    # Automatically build knowledge base if it doesn't exist
    if not (folder_path / "index.faiss").exists():
        print(f"Knowledge base for {ticker_symbol} not found. Building it now...")
        build_knowledge_base(ticker_symbol)
        print("Knowledge base built successfully.")

    # Retrieve documents and answer the question
    answer, _ = answer_question(folder_path, ticker_symbol, question)
    return answer

response = ask_question("what is the overall sentiment ?", 5)
print(response)

Knowledge base for AAPL not found. Building it now...
Knowledge base built successfully.
Based on the provided financial context, **direct evidence regarding overall market, analyst, or news sentiment is missing** [Source 1, Source 2, Source 3, Source 4, Source 5]. The retrieved text contains only historical stock price tables and balance sheet line items, with no sentiment metrics, analyst recommendations, or qualitative sentiment commentary.

---

### Observable Historical Price Trend (Derived Context)

While qualitative sentiment is not reported, the historical monthly stock price data reflects an overall upward price trend over the extended period provided:

* **Reported Facts:**
  * Earliest recorded closing price in the dataset: **$157.220001** for the period ending August 31, 2022 (`2022-08-31`) [Source 4].
  * Latest recorded closing price in the dataset: **$316.850006** for the period ending August 31, 2026 (`2026-08-31`) [Source 3].

* **Derived Calculation:**
  $$\text{Total